# OOF confidence intervals for Table 4.5 (N minus B on the 1200 out-of-fold images)

**No training. About 20-30 minutes on one T4 GPU.**

Why: in the thesis, Table 4.5 has no confidence interval for the out-of-fold (OOF) change in mAP50, instance F1 at IoU 0.5 and mAP50-95, because the OOF probability maps of B were never saved. The B fold weights (`dl3_r101_f0..4.pt`) *were* saved, so this notebook rebuilds the OOF maps of B and N by running each fold model on the 240 images it never saw during training (same folds: `StratifiedKFold(5, shuffle=True, random_state=42)`, stratified by the number of lesions capped at 3). It then scores both with the frozen balanced-DEPLOY operating points, checks that it reproduces the numbers of the ablation notebooks, and computes paired bootstrap intervals (10,000 resamples; 200 for mAP, like the test set).

**Inputs to add (same as the unified evaluation notebook):** (1) the ARCADE dataset (nirmalgaud), (2) the output of the first VG-DeepLab notebook (it holds `dl3_r101_f0..4.pt` and `vg_dl3_f0..4.pt`; or your own dataset with those 10 files). GPU = T4, Internet = On.

**Outputs (folder `oof_ci_outputs`):** `oof_paired_N_vs_B.csv` (the table to copy into the thesis), `oof_reproduction_check.csv`, `baseline_dl3_probs_oof.npz` (B OOF maps, so they are not lost again) and a zip of all of them.


In [1]:
# Do NOT let pip touch torch: --no-deps keeps the preinstalled torch exactly as it is.
!pip install -q --no-deps segmentation-models-pytorch
!pip install -q ultralytics
print("Installs complete.")

Installs complete.


In [2]:
import os, sys, re, json, math, time, shutil, zipfile, itertools, warnings, subprocess, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast
for _try in range(8):          # smp needs a few small packages; install any that are missing WITHOUT touching torch
    try:
        import segmentation_models_pytorch as smp; break
    except ModuleNotFoundError as e:
        print("missing package:", e.name, "-> installing (no deps)")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", e.name.split(".")[0].replace("_", "-")])
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
SMOKE_TEST = False         # True = quick plumbing check on 40 val + 40 test images (numbers are meaningless). False = the real run.
IMG_SIZE, STENOSIS_CATEGORY_ID = 512, 26
N_BOOT, N_BOOT_MAP = 10000, 200          # bootstrap resamples (means / mAP)
HEADLINE = "balanced-DEPLOY"
YOLO_CONF, YOLO_MAX_DET = 0.001, 20      # same as the YOLOv8s-seg notebook
CAND_THR, CAND_MIN_AREA, CAND_MAX = 0.30, 100, 8      # candidate list used for mAP of the semantic models (same as the earlier notebooks)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_CUDA = DEVICE.type == "cuda"
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if Path("/content").exists() else Path.cwd())
OUT_DIR = BASE / "oof_ci_outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [Path(p) for p in os.environ.get("UNI_WEIGHT_ROOTS", "/kaggle/input:/content/weights:/content/drive/MyDrive/arcade_stenosis_runs").split(":")]
if SMOKE_TEST: N_BOOT, N_BOOT_MAP = 500, 20
ERRORS = []
print("Torch", torch.__version__, "| smp", smp.__version__, "|", torch.cuda.get_device_name(0) if USE_CUDA else "NO GPU (enable a GPU in the notebook settings!)", "| SMOKE_TEST =", SMOKE_TEST)

# frozen operating points of the two 5-fold ensembles (chosen on out-of-fold predictions in the first VG-DeepLab run: frozen_choices_v3.json)
FROZEN_OPS = {
    "B": {"balanced-DEPLOY": dict(thr=0.40, min_area=150, tau=0.5, rel=0.75, max_k=99, top1=False),
          "maxF1-ARCADE":    dict(thr=0.40, min_area=331, tau=0.6, rel=0.75, max_k=3,  top1=True)},
    "N": {"balanced-DEPLOY": dict(thr=0.35, min_area=150, tau=0.2, rel=0.75, max_k=3,  top1=False),
          "maxF1-ARCADE":    dict(thr=0.45, min_area=150, tau=0.7, rel=0.0,  max_k=3,  top1=True)},
}

Torch 2.11.0+cu130 | smp 0.6.0 | Tesla T4 | SMOKE_TEST = False


In [3]:
# ============================================================
# ARCADE data + instance / metric code (identical logic to the VG-DeepLab and YOLOv8s-seg notebooks)
# ============================================================
def find_arcade_root():
    for root in [Path(p) for p in os.environ.get("UNI_DATA_ROOTS", "/kaggle/input").split(":")] + [Path.home() / ".cache" / "kagglehub"]:
        if root.exists():
            c = [p for p in root.rglob("train.json") if "stenosis" in str(p).lower()]
            if c: return c[0].parent.parent.parent
    return None

DATASET_ROOT = find_arcade_root()
if DATASET_ROOT is None:
    raise RuntimeError("ARCADE dataset not found. In Kaggle: Add Input -> search 'arcade-dataset' (nirmalgaud) -> Add, then Save Version -> Save & Run All.")
for s in ["train", "val", "test"]:
    assert (DATASET_ROOT / s / "annotations" / f"{s}.json").exists() and (DATASET_ROOT / s / "images").exists()
print("ARCADE root:", DATASET_ROOT)

class Inst:
    """One instance mask, cropped to its bounding box."""
    __slots__ = ("y0", "x0", "m", "area", "score", "extra")
    def __init__(self, y0, x0, m, score=1.0, extra=None):
        self.y0, self.x0, self.m = int(y0), int(x0), m
        self.area = int(m.sum()); self.score, self.extra = float(score), extra
    @property
    def y1(self): return self.y0 + self.m.shape[0]
    @property
    def x1(self): return self.x0 + self.m.shape[1]

def inst_from_full(full_bool, score=1.0):
    ys, xs = np.nonzero(full_bool)
    if len(ys) == 0: return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return Inst(y0, x0, full_bool[y0:y1, x0:x1].copy(), score)

def inst_inter(a, b):
    y0, y1 = max(a.y0, b.y0), min(a.y1, b.y1); x0, x1 = max(a.x0, b.x0), min(a.x1, b.x1)
    if y0 >= y1 or x0 >= x1: return 0
    return int(np.count_nonzero(a.m[y0 - a.y0:y1 - a.y0, x0 - a.x0:x1 - a.x0] & b.m[y0 - b.y0:y1 - b.y0, x0 - b.x0:x1 - b.x0]))

def inst_dice(a, b):
    s = a.area + b.area
    return 0.0 if s == 0 else 2.0 * inst_inter(a, b) / s

def inst_iou(a, b):
    i = inst_inter(a, b); u = a.area + b.area - i
    return 0.0 if u == 0 else i / u

def box_iou(a, b):
    ih, iw = min(a.y1, b.y1) - max(a.y0, b.y0), min(a.x1, b.x1) - max(a.x0, b.x0)
    if ih <= 0 or iw <= 0: return 0.0
    i = ih * iw; u = (a.y1 - a.y0) * (a.x1 - a.x0) + (b.y1 - b.y0) * (b.x1 - b.x0) - i
    return i / u

def paint(insts, shape):
    canvas = np.zeros(shape, bool)
    for i in insts: canvas[i.y0:i.y1, i.x0:i.x1] |= i.m
    return canvas

def greedy_match(preds, gts, fn):
    pairs = []
    for i, p in enumerate(preds):
        for j, g in enumerate(gts):
            s = fn(p, g)
            if s > 0: pairs.append((s, i, j))
    pairs.sort(key=lambda t: -t[0])
    up, ug, out = set(range(len(preds))), set(range(len(gts))), []
    for s, i, j in pairs:
        if i in up and j in ug: out.append((i, j, s)); up.discard(i); ug.discard(j)
    return out, len(up), len(ug)

def inst_greedy_match(preds, gts, criterion="dice"):
    return greedy_match(preds, gts, inst_dice if criterion == "dice" else inst_iou)

def p1_image(preds, gts):
    """Official-style ARCADE F1 for one image: matched pairs score their Dice, every unmatched prediction or lesion scores 0."""
    if not preds and not gts: return 1.0
    pairs, n_up, n_ug = inst_greedy_match(preds, gts, "dice")
    return float(np.mean([s for _, _, s in pairs] + [0.0] * (n_up + n_ug)))

def full_metrics(preds_by_img, items):
    p1, dimg, iimg = [], [], []
    TP = FP = FN = pTP = pFP = pFN = 0
    for k, it in enumerate(items):
        preds, gts = preds_by_img[k], it["gts"]
        p1.append(p1_image(preds, gts))
        pairs, n_up, n_ug = inst_greedy_match(preds, gts, "iou")
        tp = sum(1 for _, _, s in pairs if s >= 0.5)
        TP += tp; FP += len(preds) - tp; FN += len(gts) - tp
        pr, gt = paint(preds, it["shape"]), paint(gts, it["shape"])
        a, b, c = np.count_nonzero(pr & gt), np.count_nonzero(pr & ~gt), np.count_nonzero(~pr & gt)
        pTP += a; pFP += b; pFN += c
        dimg.append(1.0 if a + b + c == 0 else 2 * a / (2 * a + b + c))
        iimg.append(1.0 if a + b + c == 0 else a / (a + b + c))
    P, R = TP / max(TP + FP, 1e-9), TP / max(TP + FN, 1e-9)
    return {"P1": float(np.mean(p1)), "F1": 2 * P * R / max(P + R, 1e-9), "Precision": P, "Recall": R, "TP": TP, "FP": FP, "FN": FN,
            "Dice": 2 * pTP / max(2 * pTP + pFP + pFN, 1e-9), "IoU": pTP / max(pTP + pFP + pFN, 1e-9),
            "Dice_img": float(np.mean(dimg)), "Dice_img_std": float(np.std(dimg)), "IoU_img": float(np.mean(iimg)),
            "pred_per_img": (TP + FP) / len(items), "per_image_P1": np.array(p1), "per_image_Dice": np.array(dimg)}

_g = [Inst(0, 0, np.ones((5, 5), bool))]
assert p1_image(_g, _g) == 1.0 and abs(p1_image(_g + [Inst(9, 9, np.ones((3, 3), bool))], _g) - 0.5) < 1e-9
print("self-check official-style F1: perfect = 1.0 | one extra false positive = 0.5  -> OK")

def polys_to_mask(segs, h, w):
    full = np.zeros((h, w), np.uint8)
    for poly in segs or []:
        if isinstance(poly, list) and len(poly) >= 6:
            cv2.fillPoly(full, [np.round(np.array(poly, np.float32).reshape(-1, 2)).astype(np.int32)], 1)
    return full

def load_split(split, with_images=True):
    coco = json.load(open(DATASET_ROOT / split / "annotations" / f"{split}.json"))
    anns = {}
    for a in coco["annotations"]:
        if a["category_id"] == STENOSIS_CATEGORY_ID: anns.setdefault(a["image_id"], []).append(a)
    imgs, masks, items = [], [], []
    for im in tqdm(sorted(coco["images"], key=lambda d: d["id"]), desc=f"ARCADE {split}", mininterval=5):
        h, w = im["height"], im["width"]
        if with_images:
            g = cv2.imread(str(DATASET_ROOT / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE)
            assert g.shape == (IMG_SIZE, IMG_SIZE), f"{im['file_name']} is {g.shape}, expected 512x512 (ARCADE native)"
            imgs.append(g)
        gts, union = [], np.zeros((h, w), np.uint8)
        for a in anns.get(im["id"], []):
            m = polys_to_mask(a.get("segmentation"), h, w); union |= m
            gi = inst_from_full(m.astype(bool))
            if gi is not None: gts.append(gi)
        masks.append(union); items.append({"file": im["file_name"], "shape": (h, w), "gts": gts})
    return (np.stack(imgs) if with_images else None), np.stack(masks), items

TRAIN_IMGS, TRAIN_MASKS, TRAIN_ITEMS = load_split("train")
VAL_IMGS, VAL_MASKS, VAL_ITEMS = load_split("val")
TEST_IMGS, TEST_MASKS, TEST_ITEMS = load_split("test")
MIN_AREA_REF = int(np.percentile([g.area for it in TRAIN_ITEMS + VAL_ITEMS for g in it["gts"]], 5))   # same definition as the VG-DeepLab notebook (train+val)
# the colour read used by the original U-Net / DeepLabV3+ / YOLO notebooks must equal 3 x the grey read used here
_f = DATASET_ROOT / "test" / "images" / TEST_ITEMS[0]["file"]; _c = cv2.imread(str(_f), cv2.IMREAD_COLOR)
GRAY_OK = bool((_c[..., 0] == _c[..., 1]).all() and (_c[..., 1] == _c[..., 2]).all() and (_c[..., 0] == TEST_IMGS[0]).all())
print("images are pure grey (colour read == grey read):", GRAY_OK)
if SMOKE_TEST:
    rs = np.random.default_rng(0)
    vi = np.sort(rs.choice(len(VAL_IMGS), min(40, len(VAL_IMGS)), replace=False)); ti = np.sort(rs.choice(len(TEST_IMGS), min(40, len(TEST_IMGS)), replace=False))
    VAL_IMGS, VAL_MASKS, VAL_ITEMS = VAL_IMGS[vi], VAL_MASKS[vi], [VAL_ITEMS[i] for i in vi]
    TEST_IMGS, TEST_MASKS, TEST_ITEMS = TEST_IMGS[ti], TEST_MASKS[ti], [TEST_ITEMS[i] for i in ti]
GTU = {"val": VAL_MASKS > 0, "test": TEST_MASKS > 0}; IMGS = {"val": VAL_IMGS, "test": TEST_IMGS}; ITEMS = {"val": VAL_ITEMS, "test": TEST_ITEMS}
N_TEST = len(TEST_ITEMS)
for s in ["val", "test"]:
    n = sum(len(i["gts"]) for i in ITEMS[s]); print(f"{s:5s}: {len(ITEMS[s])} images | {n} lesions | {n / len(ITEMS[s]):.2f} per image | images without a lesion: {sum(1 for i in ITEMS[s] if not i['gts'])}")
print("5th-percentile lesion area (train+val) =", MIN_AREA_REF, "px")

ARCADE root: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/stenosis
self-check official-style F1: perfect = 1.0 | one extra false positive = 0.5  -> OK


ARCADE train:   0%|          | 0/1000 [00:00<?, ?it/s]

ARCADE val:   0%|          | 0/200 [00:00<?, ?it/s]

ARCADE test:   0%|          | 0/300 [00:00<?, ?it/s]

images are pure grey (colour read == grey read): True
val  : 200 images | 406 lesions | 2.03 per image | images without a lesion: 0
test : 300 images | 386 lesions | 1.29 per image | images without a lesion: 0
5th-percentile lesion area (train+val) = 331 px


In [4]:
# ============================================================
# Candidates, post-processing search, operating points (same code as the VG-DeepLab notebook)
# ============================================================
def cand_dict(preds, gts, gtu_k):
    preds = sorted(preds, key=lambda t: -t.score)
    D = np.zeros((len(preds), len(gts)), np.float32)
    for i, pr in enumerate(preds):
        for j, g in enumerate(gts): D[i, j] = inst_dice(pr, g)
    inter = np.array([np.count_nonzero(pr.m & gtu_k[pr.y0:pr.y1, pr.x0:pr.x1]) for pr in preds], np.int64)
    return dict(preds=preds, scores=np.array([pr.score for pr in preds]), area=np.array([pr.area for pr in preds], np.int64),
                inter=inter, D=D, gt_area=int(gtu_k.sum()), ngt=len(gts))

def build_cands(probs, gtu, items, thr, min_area):
    """probs: (N,H,W) float; one dict per image with connected components (score = mean probability) sorted by score."""
    out = []
    for k in range(len(probs)):
        p = probs[k].astype(np.float32)
        n, lab, stats, _ = cv2.connectedComponentsWithStats((p > thr).astype(np.uint8), connectivity=8)
        preds = []
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a < min_area: continue
            m = lab[y:y + h, x:x + w] == i
            preds.append(Inst(y, x, m, score=float(p[y:y + h, x:x + w][m].mean())))
        out.append(cand_dict(preds, items[k]["gts"], gtu[k]))
    return out

def keep_idx(sc, tau, rel, max_k, top1):
    n = len(sc)
    keep = [i for i in range(n) if sc[i] >= tau and sc[i] >= rel * sc[0]][:max_k]
    if not keep and top1 and n: keep = [0]
    return keep

def p1_from_D(Dsub, n_gt):
    n_pred = Dsub.shape[0]
    if n_pred == 0 and n_gt == 0: return 1.0
    scores, matched_p, matched_g = [], set(), set()
    if n_pred and n_gt:
        ii, jj = np.nonzero(Dsub > 0)
        for o in np.argsort(-Dsub[ii, jj], kind="stable"):
            i, j = int(ii[o]), int(jj[o])
            if i not in matched_p and j not in matched_g:
                scores.append(float(Dsub[i, j])); matched_p.add(i); matched_g.add(j)
    scores += [0.0] * ((n_pred - len(matched_p)) + (n_gt - len(matched_g)))
    return float(np.mean(scores))

def eval_cfg(C, tau, rel, max_k, top1):
    d_sum = p_sum = 0.0
    for c in C:
        keep = keep_idx(c["scores"], tau, rel, max_k, top1)
        P = int(c["area"][keep].sum()) if keep else 0; I = int(c["inter"][keep].sum()) if keep else 0
        d_sum += 1.0 if P + c["gt_area"] == 0 else 2.0 * I / (P + c["gt_area"])
        p_sum += p1_from_D(c["D"][keep] if keep else np.zeros((0, c["ngt"]), np.float32), c["ngt"])
    return d_sum / len(C), p_sum / len(C)

def select_insts(c, cfg):
    return [c["preds"][i] for i in keep_idx(c["scores"], cfg["tau"], cfg["rel"], cfg["max_k"], cfg["top1"])]

GRID_SEM = dict(thr=[0.30, 0.35, 0.40, 0.45, 0.50, 0.55], min_area=sorted({150, MIN_AREA_REF}),
                tau=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])
GRID_YOLO = dict(thr=[0.0], min_area=[0], tau=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])

def search_postproc(cands_fn, grid):
    """cands_fn(thr, min_area) -> candidate dicts of the VALIDATION images."""
    rows = []
    for thr, ma in itertools.product(grid["thr"], grid["min_area"]):
        C = cands_fn(thr, ma)
        for tau, rel, mk, t1 in itertools.product(grid["tau"], grid["rel"], grid["max_k"], grid["top1"]):
            d, p = eval_cfg(C, tau, rel, mk, t1)
            rows.append(dict(thr=thr, min_area=ma, tau=tau, rel=rel, max_k=mk, top1=t1, dice_img=d, p1=p))
    return pd.DataFrame(rows)

def pick_ops(df):
    ops = {}
    for tag, sub in [("ARCADE", df), ("DEPLOY", df[~df.top1])]:
        for obj, key in [("maxDice", df.dice_img), ("maxF1", df.p1), ("balanced", (df.dice_img + df.p1) / 2)]:
            r = df.loc[key.loc[sub.index].idxmax()]
            ops[f"{obj}-{tag}"] = dict(thr=float(r.thr), min_area=int(r.min_area), tau=float(r.tau), rel=float(r.rel),
                                       max_k=int(r.max_k), top1=bool(r.top1), val_dice=float(r.dice_img), val_p1=float(r.p1))
    return ops

def pixel_confusion(preds_by_img, items):
    TP = FP = FN = total = 0
    for preds, it in zip(preds_by_img, items):
        h, w = it["shape"]; total += h * w
        pr, gt = paint(preds, it["shape"]), paint(it["gts"], it["shape"])
        TP += int(np.count_nonzero(pr & gt)); FP += int(np.count_nonzero(pr & ~gt)); FN += int(np.count_nonzero(~pr & gt))
    TN = total - TP - FP - FN
    return dict(TP=TP, FP=FP, FN=FN, TN=TN, precision=TP / max(TP + FP, 1), recall=TP / max(TP + FN, 1), specificity=TN / max(TN + FP, 1),
                accuracy=(TP + TN) / max(total, 1), dice=2 * TP / max(2 * TP + FP + FN, 1), iou=TP / max(TP + FP + FN, 1))

def match_counts(preds_by_img, items, thr, fn=inst_iou):
    """Per image: (true positives at overlap >= thr, number of predictions, number of GT lesions)."""
    tp, npred, ngt = [], [], []
    for preds, it in zip(preds_by_img, items):
        pairs, _, _ = greedy_match(preds, it["gts"], fn)
        tp.append(sum(1 for _, _, s in pairs if s >= thr)); npred.append(len(preds)); ngt.append(len(it["gts"]))
    return np.array(tp), np.array(npred), np.array(ngt)

def f1_from_counts(cnt):
    tp, npred, ngt = cnt
    return 2 * tp.sum() / max(npred.sum() + ngt.sum(), 1)

def cap_cands(C, max_per_img):
    out = []
    for c in C:
        n = min(len(c["preds"]), max_per_img)
        out.append(dict(preds=c["preds"][:n], scores=c["scores"][:n], area=c["area"][:n], inter=c["inter"][:n], D=c["D"][:n], gt_area=c["gt_area"], ngt=c["ngt"]))
    return out

def map_prep(C, items):
    ious = []
    for c, it in zip(C, items):
        m = np.zeros((len(c["preds"]), len(it["gts"])), np.float32)
        for i, p in enumerate(c["preds"]):
            for j, g in enumerate(it["gts"]): m[i, j] = inst_iou(p, g)
        ious.append(m)
    return ious, np.array([len(it["gts"]) for it in items]), [np.asarray(c["scores"], np.float64) for c in C]

def map_from_prep(prep, idx=None, thrs=np.arange(0.50, 0.96, 0.05)):
    """COCO-style mask AP (101-point). idx = optional bootstrap resample of images. Returns (AP50, AP50-95)."""
    ious, ngt, scores = prep
    if idx is None: idx = np.arange(len(ngt))
    n_gt = int(ngt[idx].sum())
    order = sorted(((float(scores[k][i]), j, i) for j, k in enumerate(idx) for i in range(len(scores[k]))), key=lambda d: -d[0])
    aps = []
    for thr in thrs:
        used = [np.zeros(ngt[k], bool) for k in idx]; tp = np.zeros(len(order))
        for n, (_, j, i) in enumerate(order):
            k = idx[j]
            if ngt[k] == 0: continue
            cand = np.where(~used[j], ious[k][i], -1.0); g = int(cand.argmax())
            if cand[g] >= thr: used[j][g] = True; tp[n] = 1
        ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp); rec = ctp / max(n_gt, 1)
        prec = np.maximum.accumulate((ctp / np.maximum(ctp + cfp, 1e-9))[::-1])[::-1] if len(order) else np.zeros(0)
        aps.append(float(np.mean([prec[rec >= r].max() if np.any(rec >= r) else 0.0 for r in np.linspace(0, 1, 101)])))
    return aps[0], float(np.mean(aps))

# ---- bootstrap over test images (the same resamples for every model -> paired comparisons) ----
BI = np.random.default_rng(0).integers(0, N_TEST, (N_BOOT, N_TEST))
BI_MAP = BI[:N_BOOT_MAP]
def boot_mean(v): return v[BI].mean(1)
def boot_f1(cnt):
    tp, npred, ngt = cnt
    return 2 * tp[BI].sum(1) / np.maximum(npred[BI].sum(1) + ngt[BI].sum(1), 1)
def ci95(b): return np.percentile(b, [2.5, 97.5])
def fmt_ci(v, b): lo, hi = ci95(b); return f"{v:.4f} [{lo:.3f},{hi:.3f}]"
print("metric code ready | bootstrap resamples:", N_BOOT, "(means, F1) /", N_BOOT_MAP, "(mAP)")

metric code ready | bootstrap resamples: 10000 (means, F1) / 200 (mAP)


In [5]:
class VGNet(nn.Module):
    """5-fold models (VG-DeepLab notebook). aux: vessel channel. refine: vessel-guided refinement head. forward -> (stenosis logit, coarse logit, vessel logit or None)."""
    def __init__(self, aux=True, refine=True):
        super().__init__()
        self.aux, self.refine = aux, refine
        self.net = smp.DeepLabV3Plus(encoder_name="resnet101", encoder_weights=None, in_channels=3, classes=2 if aux else 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        if refine:
            cin = 3 if aux else 2
            self.ref = nn.Sequential(nn.Conv2d(cin, 24, 3, padding=1), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=4, dilation=4), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(24, 1, 1))
    def forward(self, x):
        out = self.net((x - self.mean) / self.std)
        s = out[:, :1]; v = out[:, 1:2] if self.aux else None
        if not self.refine: return s, s, v
        f = [x[:, :1] * 2 - 1, torch.sigmoid(s.detach()) * 2 - 1]
        if self.aux: f.append(torch.sigmoid(v.detach()) * 2 - 1)
        return s + self.ref(torch.cat(f, 1)), s, v

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6

def to_x(imgs):
    return torch.from_numpy(imgs).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)      # grey image in 3 identical channels, [0,1]


@torch.no_grad()
def predict_ms(model, imgs, bs=8):
    """5-fold models: test-time augmentation 'ms' (2 scales 1.0 / 1.25 x horizontal flip), exactly as in the VG-DeepLab notebook."""
    model.eval(); S = []
    for st in range(0, len(imgs), bs):
        x = to_x(imgs[st:st + bs]); acc = 0.0
        for sc in (1.0, 1.25):
            xs = x if sc == 1.0 else F.interpolate(x, scale_factor=sc, mode="bilinear", align_corners=False)
            for flip in (False, True):
                xi = torch.flip(xs, dims=[3]) if flip else xs
                with autocast("cuda", enabled=USE_CUDA): s2, _, _ = model(xi)
                o = torch.sigmoid(s2.float())
                if flip: o = torch.flip(o, dims=[3])
                if sc != 1.0: o = F.interpolate(o, size=x.shape[-2:], mode="bilinear", align_corners=False)
                acc = acc + o
        S.append((acc / 4)[:, 0].cpu().numpy().astype(np.float16))
    return np.concatenate(S)


In [6]:
# ============================================================
# Pool (train + val = 1200 images), the 5 folds, and the fold weights
# ============================================================
from sklearn.model_selection import StratifiedKFold
POOL_IMGS = np.concatenate([TRAIN_IMGS, VAL_IMGS]); POOL_MASKS = np.concatenate([TRAIN_MASKS, VAL_MASKS]); POOL_ITEMS = TRAIN_ITEMS + VAL_ITEMS
GTU_POOL = POOL_MASKS > 0; N_POOL = len(POOL_ITEMS)
assert N_POOL == 1200, f"expected 1200 pooled images, got {N_POOL}"
y_strat = np.array([min(len(it["gts"]), 3) for it in POOL_ITEMS])
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(N_POOL), y_strat))
print("fold sizes (held-out images):", [len(va) for _, va in FOLDS], "| lesions in pool:", sum(len(it["gts"]) for it in POOL_ITEMS))

def find_fold_ckpts(prefix):
    found = {}
    for root in [Path("/kaggle/input"), Path("/content"), BASE]:
        if root.exists():
            for p in root.rglob(f"{prefix}_f[0-4].pt"):
                f = int(p.stem.split("_f")[-1]); found.setdefault(f, p)
    assert sorted(found) == [0, 1, 2, 3, 4], f"{prefix}_f0..4.pt not all found (found folds: {sorted(found)}). Add the output of the first VG-DeepLab notebook as an input."
    return found
CK = {"B": find_fold_ckpts("dl3_r101"), "N": find_fold_ckpts("vg_dl3")}
for k, v in CK.items(): print(k, {f: str(p) for f, p in v.items()})


fold sizes (held-out images): [240, 240, 240, 240, 240] | lesions in pool: 2031
B {0: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f0.pt', 1: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f1.pt', 2: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f2.pt', 3: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f3.pt', 4: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f4.pt'}
N {0: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f0.pt', 1: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f1.pt', 2: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f2.pt', 3: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f3.pt', 4: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run

In [7]:
# ============================================================
# OOF maps: every pooled image is predicted by the fold model that did NOT train on it (TTA "ms", as in the first VG-DeepLab run)
# ============================================================
def oof_probs(key, aux, refine):
    oof = np.zeros((N_POOL, IMG_SIZE, IMG_SIZE), np.float16); t0 = time.time()
    for f, (tr, va) in enumerate(FOLDS):
        net = VGNet(aux, refine); net.load_state_dict(torch.load(CK[key][f], map_location="cpu", weights_only=True), strict=True); net.to(DEVICE).eval()
        oof[va] = predict_ms(net, POOL_IMGS[va]); del net
        if USE_CUDA: torch.cuda.empty_cache()
        print(f"{key}: fold {f + 1}/5 done ({len(va)} images) | {time.time() - t0:.0f}s")
    return oof
OOF = {"B": oof_probs("B", False, False), "N": oof_probs("N", True, True)}
np.savez_compressed(OUT_DIR / "baseline_dl3_probs_oof.npz", oof=np.round(OOF["B"].astype(np.float32) * 255).astype(np.uint8), pool_files=np.array([it["file"] for it in POOL_ITEMS]))
print("OOF maps ready.")


B: fold 1/5 done (240 images) | 29s
B: fold 2/5 done (240 images) | 51s
B: fold 3/5 done (240 images) | 75s
B: fold 4/5 done (240 images) | 98s
B: fold 5/5 done (240 images) | 122s
N: fold 1/5 done (240 images) | 31s
N: fold 2/5 done (240 images) | 61s
N: fold 3/5 done (240 images) | 91s
N: fold 4/5 done (240 images) | 122s
N: fold 5/5 done (240 images) | 153s
OOF maps ready.


In [8]:
# ============================================================
# Score B and N on the 1200 OOF images (frozen balanced-DEPLOY operating points) and check against the ablation notebooks
# ============================================================
SC = {}
for key in ["B", "N"]:
    t0 = time.time(); op = FROZEN_OPS[key][HEADLINE]
    C = build_cands(OOF[key], GTU_POOL, POOL_ITEMS, op["thr"], op["min_area"])
    preds = [select_insts(c, op) for c in C]; fm = full_metrics(preds, POOL_ITEMS); cnt = match_counts(preds, POOL_ITEMS, 0.5)
    Cmap = cap_cands(build_cands(OOF[key], GTU_POOL, POOL_ITEMS, CAND_THR, CAND_MIN_AREA), CAND_MAX)
    prep = map_prep(Cmap, POOL_ITEMS); m50, m5095 = map_from_prep(prep)
    SC[key] = dict(fm=fm, cnt=cnt, prep=prep, m50=m50, m5095=m5095, inst=float(f1_from_counts(cnt)))
    print(f"{key}: Dice/img {fm['Dice_img']:.4f} | official F1 {fm['P1']:.4f} | inst F1@0.5 {SC[key]['inst']:.4f} | mAP50 {m50:.4f} | mAP50-95 {m5095:.4f} ({time.time() - t0:.0f}s)")

# values printed by the ablation notebooks for the same OOF images (ablation_table / oof_comparison.csv)
EXPECTED_OOF = {"B": dict(Dice_img=0.5941, P1=0.4653, inst=0.4091, m50=0.2349, m5095=0.0664),
                "N": dict(Dice_img=0.6028, P1=0.4631, inst=0.4141, m50=0.2462, m5095=0.0742)}
rows, ok_all = [], True
for key in ["B", "N"]:
    got = dict(Dice_img=SC[key]["fm"]["Dice_img"], P1=SC[key]["fm"]["P1"], inst=SC[key]["inst"], m50=SC[key]["m50"], m5095=SC[key]["m5095"])
    for m, e in EXPECTED_OOF[key].items():
        ok = abs(got[m] - e) <= 0.005; ok_all &= ok
        rows.append(dict(model=key, metric=m, recorded=e, this_notebook=round(got[m], 4), difference=round(got[m] - e, 4), tolerance=0.005, status="OK" if ok else "MISMATCH"))
REPRO = pd.DataFrame(rows); REPRO.to_csv(OUT_DIR / "oof_reproduction_check.csv", index=False)
print("\n=== reproduction check (OOF) ==="); print(REPRO.to_string(index=False))
print("\nALL REPRODUCED -> the intervals below can go into Table 4.5." if ok_all else "\nWARNING: at least one number does not reproduce. Do NOT put the intervals into the thesis before the cause is understood (send me this output).")


B: Dice/img 0.5941 | official F1 0.4653 | inst F1@0.5 0.4096 | mAP50 0.2349 | mAP50-95 0.0667 (8s)
N: Dice/img 0.6028 | official F1 0.4631 | inst F1@0.5 0.4141 | mAP50 0.2462 | mAP50-95 0.0743 (8s)

=== reproduction check (OOF) ===
model   metric  recorded  this_notebook  difference  tolerance status
    B Dice_img    0.5941         0.5941     -0.0000      0.005     OK
    B       P1    0.4653         0.4653      0.0000      0.005     OK
    B     inst    0.4091         0.4096      0.0005      0.005     OK
    B      m50    0.2349         0.2349     -0.0000      0.005     OK
    B    m5095    0.0664         0.0667      0.0003      0.005     OK
    N Dice_img    0.6028         0.6028     -0.0000      0.005     OK
    N       P1    0.4631         0.4631     -0.0000      0.005     OK
    N     inst    0.4141         0.4141      0.0000      0.005     OK
    N      m50    0.2462         0.2462     -0.0000      0.005     OK
    N    m5095    0.0742         0.0743      0.0001      0.005     O

In [9]:
# ============================================================
# Paired bootstrap of N minus B over the 1200 OOF images (same resampled images for both models)
# ============================================================
BI_O = np.random.default_rng(1).integers(0, N_POOL, (N_BOOT, N_POOL)); BI_OM = BI_O[:N_BOOT_MAP]
def boot_f1_o(cnt):
    tp, npred, ngt = cnt
    return 2 * tp[BI_O].sum(1) / np.maximum(npred[BI_O].sum(1) + ngt[BI_O].sum(1), 1)
t0 = time.time()
mapB = np.array([map_from_prep(SC["B"]["prep"], idx) for idx in BI_OM]); mapN = np.array([map_from_prep(SC["N"]["prep"], idx) for idx in BI_OM])
print(f"mAP bootstrap done in {time.time() - t0:.0f}s")
tests = {
    "Dice per image":          (SC["N"]["fm"]["Dice_img"] - SC["B"]["fm"]["Dice_img"], SC["N"]["fm"]["per_image_Dice"][BI_O].mean(1) - SC["B"]["fm"]["per_image_Dice"][BI_O].mean(1)),
    "Official-style F1":       (SC["N"]["fm"]["P1"] - SC["B"]["fm"]["P1"], SC["N"]["fm"]["per_image_P1"][BI_O].mean(1) - SC["B"]["fm"]["per_image_P1"][BI_O].mean(1)),
    "Instance F1 at IoU 0.5":  (SC["N"]["inst"] - SC["B"]["inst"], boot_f1_o(SC["N"]["cnt"]) - boot_f1_o(SC["B"]["cnt"])),
    "mAP50":                   (SC["N"]["m50"] - SC["B"]["m50"], mapN[:, 0] - mapB[:, 0]),
    "mAP50-95":                (SC["N"]["m5095"] - SC["B"]["m5095"], mapN[:, 1] - mapB[:, 1]),
}
out = []
for name, (d, b) in tests.items():
    lo, hi = np.percentile(b, [2.5, 97.5])
    out.append({"metric": name, "N minus B (OOF, 1200 images)": round(d, 4), "95% CI": f"[{lo:+.4f}, {hi:+.4f}]", "CI excludes 0": "YES" if (lo > 0 or hi < 0) else "no",
                "resamples": N_BOOT_MAP if name.startswith("mAP") else N_BOOT})
TAB = pd.DataFrame(out); TAB.to_csv(OUT_DIR / "oof_paired_N_vs_B.csv", index=False)
print("\n=== Table 4.5, OOF column (copy these into the thesis) ===\n"); print(TAB.to_string(index=False))
shutil.make_archive(str(BASE / "oof_ci_outputs"), "zip", OUT_DIR)
print("\nDone. Download /kaggle/working/oof_ci_outputs.zip (Output tab) and send it to Claude.")


mAP bootstrap done in 53s

=== Table 4.5, OOF column (copy these into the thesis) ===

                metric  N minus B (OOF, 1200 images)             95% CI CI excludes 0  resamples
        Dice per image                        0.0087 [+0.0007, +0.0167]           YES      10000
     Official-style F1                       -0.0023 [-0.0121, +0.0077]            no      10000
Instance F1 at IoU 0.5                        0.0045 [-0.0133, +0.0225]            no      10000
                 mAP50                        0.0113 [-0.0060, +0.0291]            no        200
              mAP50-95                        0.0076 [+0.0013, +0.0134]           YES        200

Done. Download /kaggle/working/oof_ci_outputs.zip (Output tab) and send it to Claude.
